# For inference

- We used directories of mel spectrograms to generate sound classes predictions from SCQ-U-CNN. 
- Each class directory will result in one csv, according to testing and training/validation joined. 
- Each row of the csv corresponds to a single mel.

In [1]:
import tensorflow as tf
'''
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=1
'''
import IPython.display as display
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from uvne_packs.losses import FocalLossMultilabel


import os
import pathlib
import glob
import joblib
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
AUTOTUNE = tf.data.experimental.AUTOTUNE
tf.__version__

'2.21.0'

# 0. Inputs

In [ ]:
#output directory
out_dir = r'C:\Users\Acer\Desktop\TSI\U_VNE_CNN\5_cnn_deployment_py_R\results'

# model
SAVE_DIR_K_2 = r"C:\Users\Acer\Desktop\TSI\U_VNE_CNN\2_cnn_training-py\results\k_135_2"
BEST_MODEL_PATH_K_2 = os.path.join(SAVE_DIR_K_2, "best_model_finetuned.keras")  # U-SCQ-CNN

# data dir for images to predict class
data_path = r"C:\Users\Acer\Desktop\TSI\U_VNE_CNN\5_cnn_deployment_py_R\results\VNE_mels"

print(BEST_MODEL_PATH_K_2)
print(out_dir)

C:\Users\Acer\Desktop\TSI\U_VNE_CNN\2_cnn_training-py\results\k_135_2\best_model_finetuned.keras
C:\Users\Acer\Desktop\TSI\U_VNE_CNN\5_cnn_deployment_py_R\results


In [3]:
# dimensions of our images    
img_width, img_height, img_depth = 224, 224, 3 # 224 pixels x 224 pixels x 3 bands (RGB)

In [4]:
# number of images to predict at once 
batch_size = 50

## Functions

In [5]:
# function to read in a file path
def process_path(file_path, IMG_HEIGHT, IMG_WIDTH):
    # load the raw data from the file as a string
    img = tf.io.read_file(file_path)
    img = decode_img(img, IMG_HEIGHT, IMG_WIDTH)
    return img #, label

# function that interpretes image after being provided a path in process_path
def decode_img(img, IMG_HEIGHT, IMG_WIDTH):
  # convert the compressed string to a 3D uint8 tensor
  img = tf.image.decode_jpeg(img, channels=3)
  # Use `convert_image_dtype` to convert to floats in the [0,1] range.
  img = tf.image.convert_image_dtype(img, tf.float32)
  # resize the image to the desired size.
  return tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])

# 1. Load trained model checkpoint

In [6]:
# LOAD MODEL
model = tf.keras.models.load_model(BEST_MODEL_PATH_K_2)

In [7]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │     1,311,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 9)              │         9,225 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,336,093 (31.80 MB)

 Trainable params: 2,376,009 (9.06 MB)

 Non-trainable params: 1,208,064 (4.61 MB)

 Optimizer params: 4,752,020 (18.13 MB)

# 2. Set up sigmoid prediction

In [8]:
png_dirs = []
for filename in os.listdir(data_path):
    if os.path.isdir(os.path.join(data_path, filename)):
        png_dirs.append(filename)
if len(png_dirs) == 0: 
    png_dirs.append(data_path)

In [9]:
png_dirs

['C:\\Users\\Acer\\Desktop\\TSI\\U_VNE_CNN\\5_cnn_deployment_py_R\\results\\VNE_mels']

In [10]:
out_dir

'C:\\Users\\Acer\\Desktop\\TSI\\U_VNE_CNN\\5_cnn_deployment_py_R\\results'

In [11]:
print(BEST_MODEL_PATH_K_2)

C:\Users\Acer\Desktop\TSI\U_VNE_CNN\2_cnn_training-py\results\k_135_2\best_model_finetuned.keras


In [14]:
# iterate through each directory (png dir with mfccs within)
for i,f in enumerate(png_dirs):
    temp_dir = f
    pngs = glob.glob(os.path.join(temp_dir, "*.png"))
    temp_class = os.path.basename(os.path.normpath(temp_dir))    
    print()
    print(out_dir)
    print("Number of pngs", len(pngs)) 
    
    # check if the parent output dir exists
    if(os.path.exists(out_dir)):
        print("directory already exists. Moving to predictions...")            
    else:
        os.mkdir(out_dir)
        print("Created :", out_dir)
        
    # list to hold each dir of predictions
    sigmoid_pred_lst = []
    png_names = []
    
    # file path for class prediction
    out_path_temp = os.path.join(os.path.join(out_dir, "sigmoid",  temp_class))
    print(out_path_temp)
    
    # iterate through each melspec in file directory
    for j in range(len(pngs)):      
        temp_png = pngs[j] # png        
        png_names.append(os.path.basename(temp_png))
        
        #check if pkl (prediction) for file exists
        if os.path.isfile(out_path_temp):
            pass # skip file 
        
        # enter prediction 
        else:
            # process image
            img_ = process_path(temp_png, IMG_HEIGHT = img_height, IMG_WIDTH = img_width)
            img_ = tf.reshape(img_, shape= (1, img_height, img_width, img_depth))
            
            # get predictions
            pred = model.predict(img_, verbose=0)

            sigmoid_pred = tf.math.sigmoid(pred).numpy()
            sigmoid_pred_lst.append(sigmoid_pred)
      
    # save label preds
    flat_sigmoid = [item for sublist in sigmoid_pred_lst for item in sublist]
    df_sigmoid = pd.DataFrame(flat_sigmoid)
    df_sigmoid.columns = ["TM","MH","VO","MU","EM","SO", "BI","GE","SI"]
    
    # Agregar la columna con los nombres de los archivos PNG
    df_sigmoid['PNG_name'] = png_names
    
    # write directory csv 
    df_sigmoid.to_csv(os.path.join(out_dir, temp_class + '_VNE.csv'), index = False)     
    print('Saved this pkl at:',os.path.join(out_dir, temp_class), "\nLength was:",len(sigmoid_pred_lst))



C:\Users\Acer\Desktop\TSI\U_VNE_CNN\5_cnn_deployment_py_R\results
Number of pngs 17100
directory already exists. Moving to predictions...
C:\Users\Acer\Desktop\TSI\U_VNE_CNN\5_cnn_deployment_py_R\results\sigmoid\VNE_mels
Saved this pkl at: C:\Users\Acer\Desktop\TSI\U_VNE_CNN\5_cnn_deployment_py_R\results\VNE_mels 
Length was: 17100
